# exploracao_perdas — Auditoria de qualidade

**Escopo:** inspecionar `bronze.tb_geracao_distribuida` e `bronze.tb_inspecoes` SEM modificar tabelas. Usa as funções comuns do time e o cadastro Silver como referência de integridade.

**Fluxo:** `%run ./funcoes_comuns` → leitura → perfilamento bruto → chaves/orfandade → parse/consistência → diagnóstico final. Não executar este notebook como task obrigatória do Workflow; commitar o arquivo **sem outputs**.


In [0]:
%run ./funcoes_comuns

In [0]:
dbutils.widgets.text("catalog", "workspace")
CATALOGO = dbutils.widgets.get("catalog")
GD = ler_bronze(CATALOGO, "geracao_distribuida")
INS = ler_bronze(CATALOGO, "inspecoes")
# A Bronze atual utiliza sanitize_columns(), que transforma _source_sheet em source_sheet.
# Restauramos o nome do metadado apenas na leitura da Silver/exploração, sem alterar a Bronze.
if '_source_sheet' not in INS.columns and 'source_sheet' in INS.columns:
    INS = INS.withColumnRenamed('source_sheet', '_source_sheet')
colunas_rastreio = {'_source_sheet', '_source_file', '_ingestion_timestamp'}
faltando = colunas_rastreio - set(INS.columns)
if faltando:
    raise ValueError(f'Colunas de rastreio não encontradas em bronze.tb_inspecoes: {sorted(faltando)}; colunas atuais: {INS.columns}')
UC = spark.table(f"{CATALOGO}.silver.tb_unidades_consumidoras")
print("GERACAO DISTRIBUIDA:",GD.count(),"| INSPECOES:",INS.count(),"| UCs validas:",UC.count())
GD.printSchema(); INS.printSchema()
assert {"id_uc","potencia_kwp","data_conexao","modalidade","energia_injetada_kwh_media_mensal"}.issubset(GD.columns)
assert {"id_inspecao","id_uc","data","inspetor","resultado","tipo_irregularidade","kwh_recuperado","_source_sheet"}.issubset(INS.columns)

## 1. Perfilamento bruto de todas as colunas
A Bronze permanece intacta. Inspecionar nulos, domínios, caracteres especiais, extremidades e distribuição.

In [0]:
from pyspark.sql import functions as F

# Perfilamento sem expor unidades consumidoras ou identificadores individuais.
# O repositório será público: não persistir amostras com id_uc, id_inspecao ou inspetor.
IDENTIFICADORES_RESTRITOS = {"id_uc", "id_inspecao", "inspetor"}

for nome, df in [("geracao_distribuida", GD), ("inspecoes", INS)]:
    print("\n=====", nome.upper(), "=====")
    for coluna in df.columns:
        if coluna.startswith("_") and coluna != "_source_sheet":
            continue
        if coluna in IDENTIFICADORES_RESTRITOS:
            estat = df.agg(
                F.count("*").alias("linhas"),
                F.sum(F.col(coluna).isNull().cast("long")).alias("nulos"),
                F.countDistinct(F.col(coluna)).alias("valores_distintos"),
            ).first().asDict()
            print(f"{coluna}: resumo agregado (sem exibir valores) -> {estat}")
        else:
            perfilar_coluna(df, coluna, n=10)


## 2. Integridade, unicidade e distribuições
Comparação com a Silver de cadastro; os registros órfãos são identificados, mas não excluídos nesta exploração.

In [0]:
from pyspark.sql import functions as F
for nome,df,chave in [("GD",GD,"id_uc"),("INS",INS,"id_inspecao")]:
    print(f"{nome}: total={df.count():,}; chave duplicada={df.count()-df.select(chave).distinct().count():,}; duplicatas de negocio={df.count()-df.dropDuplicates([c for c in df.columns if not c.startswith('_')]).count():,}")
    print("Orfaos de UC:",df.join(UC.select("id_uc"),"id_uc","left_anti").count())
display(GD.groupBy("modalidade").count().orderBy(F.desc("count")))
display(INS.groupBy("_source_sheet","resultado").count().orderBy("_source_sheet","resultado"))
display(INS.groupBy("tipo_irregularidade").count().orderBy(F.desc("count")))

## 3. Conversibilidade e plausibilidade
Testa parsers compartilhados sem escrever tabelas. `checar_parse` deve falhar se algum valor não vazio deixar de ser interpretado.

In [0]:
gd_pre = aparar_textos(GD)
ins_pre = aparar_textos(INS)
for c in ["potencia_kwp","energia_injetada_kwh_media_mensal"]:
    gd_pre=gd_pre.withColumn(c,valores_para_nulo(c))
ins_pre=ins_pre.withColumn("kwh_recuperado",valores_para_nulo("kwh_recuperado"))
checar_parse(gd_pre,["potencia_kwp","energia_injetada_kwh_media_mensal"],parse_decimal,"geracao_distribuida")
checar_parse(gd_pre,["data_conexao"],parse_data,"geracao_distribuida")
checar_parse(ins_pre,["kwh_recuperado"],parse_decimal,"inspecoes")
checar_parse(ins_pre,["data"],parse_data,"inspecoes")
gd_t=gd_pre.withColumn("potencia_kwp",parse_decimal("potencia_kwp")).withColumn("energia_injetada_kwh_media_mensal",parse_decimal("energia_injetada_kwh_media_mensal")).withColumn("data_conexao",parse_data("data_conexao"))
ins_t=ins_pre.withColumn("kwh_recuperado",parse_decimal("kwh_recuperado")).withColumn("data",parse_data("data"))
display(gd_t.select("potencia_kwp","energia_injetada_kwh_media_mensal").summary())
display(ins_t.select("kwh_recuperado").summary())
print("Potencia <= 0:",gd_t.filter(F.col("potencia_kwp")<=0).count())
print("Energia injetada < 0:",gd_t.filter(F.col("energia_injetada_kwh_media_mensal")<0).count())
print("kWh recuperado < 0:",ins_t.filter(F.col("kwh_recuperado")<0).count())
print("Datas GD futuras (ref. fim do historico 2026-06-30):",gd_t.filter(F.col("data_conexao")>F.lit("2026-06-30")).count())
print("Datas INS futuras (ref. fim do historico 2026-06-30):",ins_t.filter(F.col("data")>F.lit("2026-06-30")).count())

## 4. Domínios de inspeções e relação resultado × energia
Inspecões sem irregularidade podem ter zero de recuperação legitimamente. Valores suspeitos devem ser marcados, não alterados sem evidência.

In [0]:
resultado_norm=mapear_dominio("resultado",{"CONFIRMADA":"IRREGULARIDADE CONFIRMADA","NADA CONSTA":"SEM IRREGULARIDADE"})
tipo_norm=valores_para_nulo("tipo_irregularidade")
ins_check=(ins_t.withColumn("resultado_norm",resultado_norm).withColumn("tipo_norm",tipo_norm))
display(ins_check.groupBy("resultado_norm").agg(F.count("*").alias("linhas"),F.sum((F.col("kwh_recuperado")>0).cast("int")).alias("com_recuperacao_positiva"),F.sum(F.col("tipo_norm").isNull().cast("int")).alias("tipo_nulo")))
display(ins_check.groupBy("_source_sheet").agg(F.count("*").alias("linhas"),F.min("data").alias("primeira_data"),F.max("data").alias("ultima_data")))
print("Marcador - em tipo_irregularidade:",INS.filter(F.trim(F.col("tipo_irregularidade"))=="-").count())

## 5. Conclusões preliminares e hipóteses a validar no Databricks

**Resultados da auditoria local dos arquivos originais do case (20/10/2026):**

| Achado | Contagem | Tratamento recomendado |
|---|---:|---|
| GD: unidades com potência solar extremamente acima do padrão | **35** | **FLAG** (`flag_potencia_escala_suspeita`); **NÃO dividir por 1.000 automaticamente** |
| GD: potência com vírgula decimal | **316** | `parse_decimal` / registrar conversão |
| GD: órfãos de UC e IDs repetidos | **0** | checar e criar quarentena vazia |
| Inspeções: registros em 3 abas | **1.389** | manter `_source_sheet` da Bronze |
| Inspeções: resultado com sinônimo/caixa diferente | **164** | `mapear_dominio` |
| Inspeções: resultado após padronizar | **871 confirmadas / 518 sem irregularidade** | `checar_dominio` |
| Inspeções: marcadores `-`, `NULL`, `N/A` em tipo | **77 marcadores `-` + 441 nulos de origem** | após normalização: **518 nulos**; preservar nulo, sem transformar em zero |
| Inspeções: energia recuperada com vírgula | **125** | `parse_decimal` |
| Inspeções: `inspetor` com espaços nas extremidades | **83** | `aparar_textos`, registrar no log |
| Inspeções: datas fora de ISO (hífen MM-dd-yyyy ou ponto dd.MM.yyyy) | **133** | `parse_data` e `checar_parse` |
| Inspeções: na aba `dez`, porém realizadas fora de dezembro | **107** | **FLAG** informativa; **manter data real**, não deslocar registros |
| Inspeções: órfãos de UC e IDs repetidos | **0** | checar e quarentena vazia |

**Interpretação da potência:** 35 registros com potência entre 3.049,99 e 8.994,91 kWp, em contraste com o restante (máximo de 75 kWp). A relação energia injetada mensal / potência é anormalmente baixa e fica plausível se dividida por 1.000. **Isso é hipótese de erro de escala, não confirmação.** Consultar o time antes de alterar valores; manter origem, flag e rastreio.

**Validação:** os totais desta tabela foram confrontados com as saídas efetivamente salvas da execução no Databricks. Os valores podem mudar se a origem ou as regras forem atualizadas.


## 6. Auditoria adicional: escala de potência e datas fora da aba

Essas verificações foram acrescentadas após identificar **dois problemas não destacados na versão inicial**: possível fator de escala na potência e divergência entre data real e nome da aba `dez`. Os resultados são *agregados*: não mostrar dados pessoais no repositório.


In [0]:
# Potências atípicas: preservar números da Bronze e investigar possível escala x1000.
# Os limiares são diagnósticos deste dataset (não são limites regulatórios).
from pyspark.sql import functions as F
pot = F.col('potencia_kwp')
inj = F.col('energia_injetada_kwh_media_mensal')
gd_auditoria = gd_t.withColumn(
    'flag_potencia_escala_suspeita',
    F.coalesce((pot > F.lit(1000)) & (F.when(pot > 0, inj / pot).otherwise(F.lit(None)) < F.lit(1)),F.lit(False))
)
print('Potências > 1000 kWp e injeção mensal < 1 kWh por kWp:',
      gd_auditoria.filter('flag_potencia_escala_suspeita').count())
display(gd_auditoria.groupBy('flag_potencia_escala_suspeita').agg(
    F.count('*').alias('qtd_ucs'),F.min('potencia_kwp').alias('potencia_min'),
    F.max('potencia_kwp').alias('potencia_max')))

# A aba indica agrupamento do Excel, NÃO necessariamente o período da inspeção.
mes = F.month('data')
fora_aba = (
    ((F.col('_source_sheet') == 'ago-set') & (~mes.isin(8,9))) |
    ((F.col('_source_sheet') == 'out-nov') & (~mes.isin(10,11))) |
    ((F.col('_source_sheet') == 'dez') & (mes != 12))
)
ins_auditoria = ins_t.withColumn('flag_data_fora_periodo_aba',F.coalesce(fora_aba,F.lit(False)))
display(ins_auditoria.groupBy('_source_sheet','flag_data_fora_periodo_aba').count().orderBy('_source_sheet','flag_data_fora_periodo_aba'))
